# Run all notebooks
This notebook executes all notebooks under `EXCEL/` and saves the executed outputs back into each `.ipynb` file.

**How to use**
1. Open this notebook
2. Run Cell 2

**Notes**
- Uses `nbclient` + `nbformat` to run notebooks.
- Each notebook runs in its own kernel (clean state).
- Skips `.ipynb_checkpoints`, hidden folders, and `EXCEL/Data/`.

In [2]:
from __future__ import annotations

from pathlib import Path

def find_workspace_root(start: Path) -> Path:
    start = start.resolve()
    for p in (start, *start.parents):
        if (p / 'EXCEL').exists():
            return p
    return start

ROOT = find_workspace_root(Path.cwd())
EXCEL_DIR = ROOT / 'EXCEL'

EXCLUDE_DIRNAMES = {'.ipynb_checkpoints', '.git', '__pycache__', 'Data'}
THIS_NOTEBOOK_NAME = 'Run_All.ipynb'

KERNEL_NAME = 'python3'
TIMEOUT_SECONDS = 60 * 60  # 1 hour per notebook
ALLOW_ERRORS = False       # set True if you want it to continue on errors

def iter_notebooks(base: Path) -> list[Path]:
    paths: list[Path] = []
    if not base.exists():
        return paths
    for p in base.rglob('*.ipynb'):
        if any(part in EXCLUDE_DIRNAMES for part in p.parts):
            continue
        if any(part.startswith('.') for part in p.parts):
            continue
        if p.name == THIS_NOTEBOOK_NAME:
            continue
        paths.append(p)
    return sorted(set(paths), key=lambda x: (str(x.parent), x.name))


def run_notebook(path: Path) -> None:
    try:
        import nbformat
        from nbclient import NotebookClient
    except ModuleNotFoundError:
        import sys
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'nbclient', 'nbformat'])
        import nbformat
        from nbclient import NotebookClient

    nb = nbformat.read(path, as_version=4)
    client = NotebookClient(
        nb,
        kernel_name=KERNEL_NAME,
        timeout=TIMEOUT_SECONDS,
        allow_errors=ALLOW_ERRORS,
        resources={"metadata": {"path": str(path.parent)}},
    )
    client.execute()
    nbformat.write(nb, path)


notebooks = iter_notebooks(EXCEL_DIR)
print(f'Workspace root: {ROOT}')
print(f'Found {len(notebooks)} notebooks to run under EXCEL/.')
for i, p in enumerate(notebooks, start=1):
    rel = p.relative_to(ROOT)
    print(f'[{i:02d}/{len(notebooks):02d}] Running: {rel} ...')
    run_notebook(p)
print('Done.')

Found 9 notebooks to run under EXCEL/.
[01/09] Running: EXCEL/Alder.ipynb ...
[02/09] Running: EXCEL/Arbejdstidsfordeling.ipynb ...
[03/09] Running: EXCEL/Figur1+2.ipynb ...
[04/09] Running: EXCEL/Frivilligt_ekstraarbejde.ipynb ...
[05/09] Running: EXCEL/Merarbejde.ipynb ...
[06/09] Running: EXCEL/Sygdom.ipynb ...
[07/09] Running: EXCEL/arbejdstid.ipynb ...
[08/09] Running: EXCEL/køn.ipynb ...
[09/09] Running: EXCEL/løn.ipynb ...
Done.
